# Neural network vs. the random forest

The project's winning model is a random forest — 37 columns, `ps_calc_*`
dropped — at **Gini 0.27244 ± 0.00305** (#13), confirmed once against the untouched holdout at
**0.27308** (#24). Nothing has yet tested whether a neural network does better.

This notebook answers that, and prices it. It produces:

1. a reframing of the problem for a NN, including what makes this dataset a stretch for one
2. a baseline feedforward network scored on the **identical five folds** as the forest
3. a light tuning pass, with training and inference time measured, not estimated
4. a seven-dimension comparison table
5. the inputs to the verdict written up in `README.md`

**The holdout is never loaded.** `load_final_test` is never *called* here — the only occurrences
of that name in this notebook are this sentence and the comment in the loading cell below.
Everything happens on the 80% training split through the shared harness in `src/evaluation.py`,
which is imported and never modified — so every existing row in `results.md` stays comparable.

In [1]:
# Same repo-root preamble as notebooks 01-04: evaluation.py uses a relative
# data path, so the working directory matters.
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import math
import time
import numpy as np
import torch
import torch.nn as nn
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from data_loading import feature_groups
from evaluation import N_FOLDS, SEED, cross_validate_model, gini_normalized, load_train
from train_baseline_models import _build_preprocessor, _check_realistic

# Determinism over speed. Measured on this machine: CPU is bitwise reproducible
# across processes AND across thread counts, while MPS produces a different
# answer from CPU -- which would make the device part of the recorded config and
# put every number here out of reach of a teammate on non-Apple hardware.
# MPS was also only ~20% faster on a 2-minute job.
torch.use_deterministic_algorithms(True)

print(f"torch {torch.__version__} | numpy {np.__version__}")
print(f"cwd {Path.cwd()}")

torch 2.14.0 | numpy 2.5.2
cwd /Users/fritzvonscherzer/Documents/AIPM/porto-seguro-ml-project


---
## 1. Reframing the problem for a neural network

### 1.1 Problem type

Binary classification — did this policyholder file a claim — but scored as a **ranking**.
The KPI is normalized Gini (`2·AUC − 1`), which depends only on the *order* of the predicted
scores, never their magnitudes. `src/evaluation.py` asserts exactly that property.

Two consequences for the architecture, both of which fall straight out:

- **Output layer:** a single logit, with `BCEWithLogitsLoss`. Not softmax over two classes —
  that would be the same model with twice the output parameters and a redundant degree of freedom.
- **The sigmoid doesn't matter.** It's applied only at scoring time, and it's monotonic, so it
  cannot change the ranking or therefore the Gini. We could score on the raw logits and get an
  identical number. We apply it anyway so `predict_proba` means what sklearn expects.

### 1.2 Input shape

Same 37 columns the winning forest uses (every `ps_calc_*` dropped, per #13). Using all 57 would
confound *model choice* with *column choice* — the comparison would stop being about the model.
The cell below confirms the shape rather than asserting it from memory.

In [2]:
train = load_train()          # the 80% split; load_final_test() is never called in this notebook
cat_cols, qty_cols = feature_groups(train.columns)
cat_37 = [c for c in cat_cols if "_calc_" not in c]
qty_37 = [c for c in qty_cols if "_calc_" not in c]

X = train[cat_37 + qty_37].values     # column order is load-bearing: the
y = train["target"].values            # preprocessor selects by POSITION, not name
BASE_RATE = y.mean()

print(f"rows              {len(train):,}")
print(f"claims            {int(y.sum()):,}  ({BASE_RATE:.4%})")
print(f"source columns    {X.shape[1]}  ({len(cat_37)} categorical + {len(qty_37)} quantity)")

# What the network actually sees, after the same preprocessing the logistic
# regression got: median-impute the -1 sentinel, one-hot categoricals, standardize
# quantities. Fit on ALL rows, not a subset: ps_car_11_cat has 104 levels and
# several are rare, so a 50k-row sample misses some and under-reports the width.
_probe = _build_preprocessor(len(cat_37), len(qty_37), scale=True).fit_transform(X)
N_FEATURES = _probe.shape[1]
print(f"input features    {N_FEATURES}  (after one-hot + impute + standardize)")
print(f"dtype emitted     {_probe.dtype}  <- float64; the network casts to float32")
del _probe

rows              475,967
claims            17,461  (3.6685%)
source columns    37  (25 categorical + 12 quantity)


input features    206  (after one-hot + impute + standardize)
dtype emitted     float64  <- float64; the network casts to float32


### 1.3 What makes a neural network a stretch here

Worth stating plainly before we build anything, because it sets the expectation the results
should be read against.

| | Why it hurts a NN |
|---|---|
| **No spatial or sequential structure** | The inductive biases that make networks win on images and text — weight sharing, locality, translation invariance — have nothing to attach to. Column 14 is not "next to" column 15 in any meaningful sense. A dense net has to learn from scratch what a tree gets from its splitting rule for free. |
| **Only ~17.5k claims** | The 80% split has 475,967 rows but just **17,461** positives. Deep learning's advantage shows up with abundant labelled signal; here the effective sample size for the thing we care about is small. |
| **Anonymized features** | `ps_ind_03`, `ps_car_13` — no domain meaning, so no way to shape an architecture around what the inputs represent. Feature crosses, embeddings and grouping all become guesswork. |
| **`ps_car_11_cat` has 104 levels** | One-hot turns it into 104 sparse columns, several at ~0.1% density. Trees split cleanly on rare levels; dense layers have to allocate weights to columns that are almost always zero. |
| **`-1` is informative missingness** | A tree can split on "is this missing" natively. We median-impute it away for the network, destroying signal that #22 already showed carries a little information. |
| **The prior from this competition** | Gradient-boosted trees dominated the leaderboard. The known neural-network results appeared as *components of ensembles*, not as standalone winners. |

None of that means "don't try." It means the honest expected outcome is **roughly par, at higher
cost** — and that's a finding worth having measured rather than assumed.

### 1.4 Why PyTorch rather than TensorFlow/Keras

- **The harness already owns the training loop's job.** `cross_validate_model()` decides folds,
  fitting and scoring. PyTorch is eager Python, so wrapping it in sklearn's `fit`/`predict_proba`
  contract is ~60 lines. Keras would bring a second, competing abstraction — `model.compile`,
  `model.fit`, callbacks, its own metric objects — that duplicates what we already have and would
  need to be bypassed to keep the comparison fair.
- **We need none of Keras' batteries.** Its value is highest for callbacks, data generators and
  distribution strategies. At 206 features and ~0.6s/epoch on one CPU core, none apply.
- **Install cost.** `torch` is a single 121MB CPU/MPS wheel on Apple Silicon. The TensorFlow
  story on macOS (`tensorflow-macos` + `tensorflow-metal`) is heavier and more fragile for a
  `uv.lock` the whole team shares.
- **Determinism is explicit.** `results.md` requires numbers that reproduce. PyTorch lets us pin
  seeding, thread count and algorithm choice directly, and verify it (we do, below).

This is a judgement about *fit to this codebase*, not a claim that one framework trains better
networks. On the modelling itself they are interchangeable here.

---
## 2. The baseline network

A plain feedforward net — no CNN, no RNN, no transformer, because as section 1.3 says there is no
structure in this data for any of them to exploit.

```
204-ish inputs → Linear(128) → ReLU → Dropout(0.3) → [→ Linear(64) → ReLU → Dropout(0.3)] → Linear(1)
```

One hidden layer or two — the bracketed block is the architecture variant, and the tuning pass in
3.2 decides which. Adam, `BCEWithLogitsLoss`. The interesting part isn't the architecture — it's making the thing
behave like a scikit-learn estimator so it can go through the **exact same** `cross_validate_model()`
as the forest. That harness calls `clone(model)`, then `fit()` on a raw numpy array, then
`predict_proba(...)[:, 1]`. sklearn's `clone()` contract is strict and most violations fail
*silently*, so the comments below mark the traps.

In [3]:
# Untrained loss is ~0.16 with the output-bias init, ~0.69 without. Anything
# past this is divergence, not a merely-bad fit.
DIVERGED_LOSS = 10.0


class TorchMLPClassifier(ClassifierMixin, BaseEstimator):
    """
    A feedforward neural network that behaves like a scikit-learn classifier.

    ClassifierMixin comes FIRST in the bases: ClassifierMixin.__sklearn_tags__
    calls super().__sklearn_tags__() and then sets estimator_type="classifier".
    With BaseEstimator first, that tag silently never gets set.

    Everything __init__ receives is stored verbatim under its own name and
    nothing else happens here. sklearn's clone() rebuilds the estimator from
    get_params() and then checks that each parameter is the SAME OBJECT it
    passed in -- so `self.hidden = list(hidden)` or
    `self.device = torch.device(device)` raises RuntimeError, and building the
    network here would draw its initial weights off the global RNG at clone
    time, giving fold 3 different starting weights than fold 1 for reasons
    nobody can see. All of that belongs in fit().
    """

    def __init__(self, hidden=(128, 64), lr=1e-3, epochs=6, batch_size=1024,
                 dropout=0.3, weight_decay=1e-4, pos_weight=None,
                 init_output_bias=True, random_state=SEED, n_threads=1,
                 eval_set=None):
        self.hidden = hidden
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.dropout = dropout
        self.weight_decay = weight_decay
        self.pos_weight = pos_weight
        self.init_output_bias = init_output_bias
        self.random_state = random_state
        self.n_threads = n_threads
        self.eval_set = eval_set

    # -- internals ---------------------------------------------------------

    def _build(self, n_features, base_rate):
        layers, prev = [], n_features
        for width in self.hidden:
            layers += [nn.Linear(prev, width), nn.ReLU(), nn.Dropout(self.dropout)]
            prev = width
        out = nn.Linear(prev, 1)
        if self.init_output_bias:
            # Start the output where the base rate already is, instead of
            # spending the first few hundred steps driving one scalar from
            # 0 down to -3.27 while the weight gradients are tiny.
            with torch.no_grad():
                out.bias.fill_(float(np.log(base_rate / (1 - base_rate))))
        layers.append(out)
        return nn.Sequential(*layers)

    @staticmethod
    def _as_float32(X):
        if not isinstance(X, np.ndarray):
            X = np.asarray(X)
        # The ColumnTransformer emits float64 (np.hstack upcasts the float32
        # one-hot block against the float64 quantity block). nn.Linear is
        # float32, so cast here rather than doubling the cost of every matmul.
        return np.ascontiguousarray(X, dtype=np.float32)

    # -- sklearn API -------------------------------------------------------

    def fit(self, X, y):
        X = self._as_float32(X)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        assert len(self.classes_) == 2, "binary classification only"
        yb = (y == self.classes_[1]).astype(np.float32)
        self.n_features_in_ = X.shape[1]

        prev_threads = torch.get_num_threads()
        try:
            torch.set_num_threads(self.n_threads)
            # fork_rng restores the process-global RNG on exit, so fitting a
            # model is not a hidden source of nondeterminism for anything
            # that runs after it.
            with torch.random.fork_rng(devices=[], device_type="cpu"):
                torch.manual_seed(self.random_state)
                gen = torch.Generator().manual_seed(self.random_state)

                net = self._build(self.n_features_in_, float(yb.mean()))
                opt = torch.optim.Adam(net.parameters(), lr=self.lr,
                                       weight_decay=self.weight_decay)
                pw = (None if self.pos_weight is None
                      else torch.tensor([float(self.pos_weight)]))
                lossf = nn.BCEWithLogitsLoss(pos_weight=pw)

                Xt = torch.from_numpy(X)
                yt = torch.from_numpy(yb)
                n, bs = len(Xt), self.batch_size
                self.epoch_losses_ = []

                # Optional per-epoch scoring, used ONLY by the tuning phase to
                # draw a validation curve. eval_set always comes from rows
                # inside the fold's own training data -- never an outer fold.
                self.val_curve_ = []
                ev = self.eval_set
                if ev is not None:
                    Xe = torch.from_numpy(self._as_float32(ev[0]))
                    ye = np.asarray(ev[1])

                net.train()
                t0 = time.perf_counter()
                for _ in range(self.epochs):
                    # Permute once and slice contiguously. Per-batch fancy
                    # indexing (Xt[perm[i:i+bs]]) costs ~3x more than the
                    # training itself.
                    perm = torch.randperm(n, generator=gen)
                    Xs, ys = Xt[perm], yt[perm]
                    total = 0.0
                    for i in range(0, n, bs):
                        opt.zero_grad(set_to_none=True)
                        loss = lossf(net(Xs[i:i + bs]).squeeze(1), ys[i:i + bs])
                        loss.backward()
                        opt.step()
                        total += loss.detach().item() * len(ys[i:i + bs])
                    epoch_loss = total / n
                    # A non-finite check alone is NOT enough: BCEWithLogitsLoss
                    # is log-sum-exp stable, so a catastrophically diverged run
                    # returns a large FINITE loss (measured: ~19,000 at lr=100)
                    # and would be silently recorded as a Gini near zero.
                    # An untrained model here sits at ~0.16 (with the bias init)
                    # to ~0.69 (without), so anything above DIVERGED_LOSS is
                    # unambiguously broken rather than merely bad.
                    if not np.isfinite(epoch_loss) or epoch_loss > DIVERGED_LOSS:
                        raise FloatingPointError(
                            f"training diverged (loss={epoch_loss:.3g}) at lr={self.lr} "
                            "-- a diverged run must not be silently recorded"
                        )
                    self.epoch_losses_.append(epoch_loss)

                    if ev is not None:
                        net.eval()
                        with torch.no_grad():
                            pe = net(Xe).squeeze(1).sigmoid().numpy()
                        self.val_curve_.append(gini_normalized(ye, pe))
                        net.train()
                self.fit_seconds_ = time.perf_counter() - t0
                self.module_ = net.eval()
        finally:
            torch.set_num_threads(prev_threads)
        return self

    def predict_proba(self, X, batch_size=8192):
        X = self._as_float32(X)
        # .eval() defensively: if dropout were left active at inference it
        # would degrade the ranking AND consume global RNG, so two identical
        # predict calls would disagree.
        self.module_.eval()
        out = np.empty(len(X), dtype=np.float64)
        with torch.no_grad():
            Xt = torch.from_numpy(X)
            for i in range(0, len(Xt), batch_size):
                out[i:i + batch_size] = (
                    self.module_(Xt[i:i + batch_size]).squeeze(1).sigmoid().numpy()
                )
        assert np.isfinite(out).all(), "non-finite probabilities"
        return np.column_stack([1.0 - out, out])

    def predict(self, X):
        return self.classes_[(self.predict_proba(X)[:, 1] > 0.5).astype(int)]

### 2.1 Correctness checks

`results.md` only accepts numbers that reproduce, so the estimator has to earn that before it
produces any. These mirror the AC-style checks in `src/evaluation.py` and
`train_baseline_models.main()`.

In [4]:
print("CHECK 1  clone() round-trip")
est = TorchMLPClassifier(hidden=(128, 64), lr=3e-4)
c = clone(est)
assert c.get_params() == est.get_params(), "params did not round-trip"
# The real trap: clone() rebuilds via klass(**params) and then checks that
# each attribute IS the object it just passed in. Any coercion in
# __init__ (list(hidden), torch.device(device)) fails that check. Prove
# the constructor is clean by round-tripping its own output.
assert TorchMLPClassifier(**est.get_params()).get_params() == est.get_params()
print("         PASS\n")

print("CHECK 2  works inside a Pipeline, predict_proba visible pre-fit")
pipe = make_pipeline(StandardScaler(), TorchMLPClassifier())
clone(pipe)
assert hasattr(pipe, "predict_proba"), "harness would take the wrong branch"
print("         PASS\n")

print("CHECK 3  reproducible across runs, global RNG perturbed in between")
rng = np.random.default_rng(0)
Xs = rng.normal(size=(6000, 20)).astype(np.float64)
ys = (rng.random(6000) < 0.1).astype(int)
m = TorchMLPClassifier(hidden=(16,), epochs=2, batch_size=512)
a = cross_validate_model(m, Xs, ys, n_folds=3, verbose=False)[2]
torch.randn(1000); np.random.rand(10)   # deliberately disturb global state
b = cross_validate_model(m, Xs, ys, n_folds=3, verbose=False)[2]
print(f"         run A {a}")
print(f"         run B {b}")
assert np.array_equal(a, b), "not reproducible"
print("         PASS (bitwise identical)\n")

print("CHECK 4  predict_proba is stable across calls")
m2 = TorchMLPClassifier(hidden=(16,), epochs=2, batch_size=512).fit(Xs, ys)
assert np.array_equal(m2.predict_proba(Xs), m2.predict_proba(Xs))
print("         PASS\n")

print("CHECK 5  divergence guard fires")
try:
    TorchMLPClassifier(hidden=(16,), epochs=3, lr=100.0).fit(Xs, ys)
    raise AssertionError("guard did NOT fire")
except FloatingPointError as e:
    print(f"         raised as expected: {str(e)[:60]}...")
print("         PASS\n")

print("CHECK 6  fork_rng leaves the global RNG untouched")
torch.manual_seed(7); before = torch.randn(3)
torch.manual_seed(7)
TorchMLPClassifier(hidden=(8,), epochs=1, batch_size=2048).fit(Xs, ys)
after = torch.randn(3)
assert torch.equal(before, after), "fit() leaked RNG state"
print("         PASS\n")

CHECK 1  clone() round-trip
         PASS

CHECK 2  works inside a Pipeline, predict_proba visible pre-fit
         PASS

CHECK 3  reproducible across runs, global RNG perturbed in between
         run A [ 0.01610325 -0.03985569  0.04941147]
         run B [ 0.01610325 -0.03985569  0.04941147]
         PASS (bitwise identical)

CHECK 4  predict_proba is stable across calls


         PASS

CHECK 5  divergence guard fires
         raised as expected: training diverged (loss=1.82e+04) at lr=100.0 -- a diverged ...
         PASS

CHECK 6  fork_rng leaves the global RNG untouched
         PASS



---
## 3. Training, tuning and evaluation

### 3.1 The fairness problem, and how it's handled

The network needs a validation set to decide when to stop training. The forest did not — it has no
equivalent knob, and `n_estimators=200` was simply fixed up front.

That asymmetry is a trap. Carving 15% out of each fold's training rows for validation would leave
the network training on **~324k rows** while the forest trained on **~381k**. Any gap in the final
number would then be partly a model difference and partly a *data* difference, and we would have no
way to separate them.

So epochs is treated as a hyperparameter fixed a priori — exactly like `n_estimators=200`:

| phase | what it uses | what it decides |
|---|---|---|
| **tuning** | an 85/15 split taken *inside the fold's own training rows* | the epoch budget, learning rate, batch size, architecture |
| **recorded run** | the **full** fold-training set, no inner split | nothing — the budget is already fixed |

No fold's validation rows influence the model scored against them, and at scoring time the network
and the forest see **identical training rows**.

Two refinements that turned out to matter:

**The budget is fixed in optimizer steps, not epochs.** An epoch over 381k rows is 1.18× more
gradient steps than an epoch over 324k. Since the validation curve peaks early and then decays,
over-stepping is the harmful direction — so we convert the tuned epoch count into a step count and
back again.

**The budget is picked from a smoothed average of many runs, not a single argmax.** Individual
epoch-to-epoch differences here are noise; picking the single best epoch off one curve would be
fitting that noise.

One thing worth saying out loud, since it cuts the other way: the 37-column set was itself chosen
by maximising CV Gini *on these same five folds* (#13). That is direct selection on the scored
partition. So epoch selection biases the network's number **less** than column selection already
biases the forest's — the comparison is, if anything, tilted toward the incumbent.

### 3.2 Light tuning

Learning rate, batch size, epochs, and one architecture variant (one hidden layer vs two) — the
scope agreed for this ticket. Everything here runs on **inner splits only**.

Each configuration is run across 2 outer folds × 2 seeds = 4 curves, averaged, then smoothed with a
3-epoch rolling mean. Two numbers come out per config: where the smoothed curve peaks, and how much
the curve moves after epoch 3 (**tail-spread**).

Tail-spread is the one to watch. A flat curve means "train for N epochs" is a safe instruction; a
peaked curve means the recorded number depends on guessing N correctly, and we have already
established that N cannot be guessed to better than noise. **This takes ~11 minutes.**

In [5]:
MAX_EPOCHS, TUNE_FOLDS, TUNE_SEEDS = 25, 2, (42, 43)

splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
outer = list(splitter.split(X, y))

# Build the inner splits once. For each tuning fold, take that fold's TRAINING
# rows and cut 85/15. The fold's own validation rows are never touched here --
# they belong to the scoring phase in 3.3.
inner = []
for f in range(TUNE_FOLDS):
    tr_idx, _ = outer[f]
    pre = _build_preprocessor(len(cat_37), len(qty_37), scale=True)
    itr, iva = train_test_split(tr_idx, test_size=0.15, random_state=SEED,
                                stratify=y[tr_idx])
    inner.append((pre.fit_transform(X[itr]), y[itr], pre.transform(X[iva]), y[iva]))
    print(f"fold {f}: inner-train {len(itr):,} rows ({int(y[itr].sum()):,} claims)  "
          f"inner-val {len(iva):,} rows ({int(y[iva].sum()):,} claims)")

GRID = [dict(hidden=h, lr=lr, batch_size=bs)
        for h in [(128, 64), (128,)] for lr in [1e-3, 3e-4] for bs in [1024, 4096]]

def smooth(v, w=3):
    return np.array([float(np.mean(v[max(0, i - w // 2):min(len(v), i + w // 2 + 1)]))
                     for i in range(len(v))])

print(f"\n{'architecture':<12}{'lr':>8}{'batch':>7}{'epoch':>7}{'peak':>10}"
      f"{'tail-spread':>13}{'seed-sd':>9}")
tuning = []
t_tune = time.perf_counter()
for cfg in GRID:
    curves = np.array([
        TorchMLPClassifier(**cfg, dropout=0.3, weight_decay=1e-4, epochs=MAX_EPOCHS,
                           random_state=sd, eval_set=(Xva, yva)).fit(Xtr, ytr).val_curve_
        for (Xtr, ytr, Xva, yva) in inner for sd in TUNE_SEEDS])
    mean_curve = curves.mean(axis=0)
    sm = smooth(mean_curve)
    best_ep = int(np.argmax(sm)) + 1
    tail = mean_curve[2:]                      # flatness from epoch 3 onward
    rec = dict(cfg=cfg, epoch=best_ep, peak=float(sm[best_ep - 1]),
               spread=float(tail.max() - tail.min()),
               seed_sd=float(curves[:, best_ep - 1].std()), curve=mean_curve)
    tuning.append(rec)
    print(f"{str(cfg['hidden']):<12}{cfg['lr']:>8g}{cfg['batch_size']:>7d}"
          f"{best_ep:>7d}{sm[best_ep-1]:>+10.5f}{rec['spread']:>13.4f}{rec['seed_sd']:>9.4f}")
print(f"\ntuning took {time.perf_counter() - t_tune:.0f}s")

fold 0: inner-train 323,657 rows (11,874 claims)  inner-val 57,116 rows (2,095 claims)


fold 1: inner-train 323,657 rows (11,873 claims)  inner-val 57,116 rows (2,095 claims)

architecture      lr  batch  epoch      peak  tail-spread  seed-sd


(128, 64)      0.001   1024     10  +0.27263       0.0091   0.0066


(128, 64)      0.001   4096     15  +0.27360       0.0075   0.0080


(128, 64)     0.0003   1024     10  +0.27545       0.0100   0.0083


(128, 64)     0.0003   4096     23  +0.27574       0.0239   0.0099


(128,)         0.001   1024     10  +0.27521       0.0057   0.0079


(128,)         0.001   4096     15  +0.27556       0.0100   0.0077


(128,)        0.0003   1024     14  +0.27619       0.0119   0.0085


(128,)        0.0003   4096     25  +0.27607       0.0250   0.0093

tuning took 477s


In [6]:
# Choosing the winner. The peaks are separated by far less than the seed noise,
# so ranking on peak height alone would be ranking on noise. Among configs whose
# peak is within one seed-sd of the best, take the FLATTEST curve -- that is the
# one for which "train for N epochs" is a safe instruction rather than a bet.
best_peak = max(t["peak"] for t in tuning)
tolerance = max(t["seed_sd"] for t in tuning)
contenders = [t for t in tuning if t["peak"] >= best_peak - tolerance]
WINNER = min(contenders, key=lambda t: t["spread"])

print(f"best peak overall      {best_peak:+.5f}")
print(f"seed noise (tolerance) {tolerance:.5f}  -> {len(contenders)} of {len(tuning)} "
      f"configs are statistically tied")
print(f"\nchosen (flattest of the tied): {WINNER['cfg']}")
print(f"  smoothed peak {WINNER['peak']:+.5f}   tail-spread {WINNER['spread']:.4f}"
      f"   epoch {WINNER['epoch']}")

BEST_CFG = dict(WINNER["cfg"], dropout=0.3, weight_decay=1e-4)
TUNED_EPOCHS = WINNER["epoch"]

best peak overall      +0.27619
seed noise (tolerance) 0.00994  -> 8 of 8 configs are statistically tied

chosen (flattest of the tied): {'hidden': (128,), 'lr': 0.001, 'batch_size': 1024}
  smoothed peak +0.27521   tail-spread 0.0057   epoch 10


### 3.3 Converting the epoch budget

The tuned budget was measured on 324k inner-training rows. The recorded run trains on the full
381k, where each epoch is 1.18× more gradient steps. We convert through steps so the network gets
the same amount of *optimization*, not the same number of passes.

In [7]:
TUNE_N, FOLD_TRAIN_N = len(inner[0][1]), len(outer[0][0])
BS = BEST_CFG["batch_size"]
steps = TUNED_EPOCHS * math.ceil(TUNE_N / BS)
EPOCHS = round(steps / math.ceil(FOLD_TRAIN_N / BS))

print(f"tuned on      {TUNE_N:,} rows -> {math.ceil(TUNE_N/BS)} steps/epoch"
      f" x {TUNED_EPOCHS} epochs = {steps} steps")
print(f"scoring on    {FOLD_TRAIN_N:,} rows -> {math.ceil(FOLD_TRAIN_N/BS)} steps/epoch")
print(f"budget        {EPOCHS} epochs  ({EPOCHS * math.ceil(FOLD_TRAIN_N/BS)} steps)")

def nn_pipe(seed=SEED, **over):
    return make_pipeline(
        _build_preprocessor(len(cat_37), len(qty_37), scale=True),
        TorchMLPClassifier(**{**BEST_CFG, **over}, epochs=EPOCHS, random_state=seed))

tuned on      323,657 rows -> 317 steps/epoch x 10 epochs = 3170 steps
scoring on    380,773 rows -> 372 steps/epoch
budget        9 epochs  (3348 steps)


### 3.4 The recorded run

Five folds × **three seeds**. The third dimension is not optional here: measured seed-to-seed
spread at a fixed epoch is larger than the forest's entire fold std, so a single-seed number could
not be honestly compared against 0.27244 ± 0.00305. The headline is the mean of the three CV means.

Guards, matching what the rest of the project already does: `_check_realistic()` blocks anything at
or above the 0.30 leak ceiling, a floor catches a silently broken pipeline, and the whole run is
repeated to confirm the numbers reproduce bitwise before any of them are written down.

In [8]:
per_seed = {}
t_all = time.perf_counter()
for sd in (42, 43, 44):
    t0 = time.perf_counter()
    m, s, sc = cross_validate_model(nn_pipe(seed=sd), X, y, verbose=False)
    per_seed[sd] = dict(mean=m, std=s, scores=sc, runtime=time.perf_counter() - t0)
    print(f"seed {sd}:  {m:+.5f} +/- {s:.5f}   "
          f"folds {[f'{v:+.5f}' for v in sc]}  ({per_seed[sd]['runtime']:.0f}s)")
    _check_realistic(f"NN seed {sd}", m)
    assert m > 0.15, "implausibly low -- debug the pipeline before recording"

NN_RUNTIME = time.perf_counter() - t_all
means = [per_seed[s]["mean"] for s in (42, 43, 44)]
NN_MEAN, NN_SEED_SD = float(np.mean(means)), float(np.std(means))
NN_FOLD_SD = float(np.mean([per_seed[s]["std"] for s in (42, 43, 44)]))

print(f"\nheadline Gini      {NN_MEAN:+.5f}")
print(f"seed spread        {NN_SEED_SD:.5f}   (across the 3 CV means)")
print(f"mean fold std      {NN_FOLD_SD:.5f}")
print(f"runtime            {NN_RUNTIME:.0f}s for 3 seeds ({NN_RUNTIME/3:.0f}s per CV run)")

seed 42:  +0.27900 +/- 0.00672   folds ['+0.26691', '+0.28451', '+0.28064', '+0.27733', '+0.28562']  (36s)


seed 43:  +0.27780 +/- 0.00674   folds ['+0.27159', '+0.28936', '+0.27300', '+0.27354', '+0.28151']  (35s)


seed 44:  +0.27723 +/- 0.00689   folds ['+0.26644', '+0.28791', '+0.27934', '+0.27597', '+0.27650']  (35s)

headline Gini      +0.27801
seed spread        0.00074   (across the 3 CV means)
mean fold std      0.00678
runtime            106s for 3 seeds (35s per CV run)


In [9]:
# Reproducibility gate. train_baseline_models.main() does this for the forest;
# a number that does not reproduce does not go in results.md.
_, _, again = cross_validate_model(nn_pipe(seed=42), X, y, verbose=False)
identical = np.array_equal(per_seed[42]["scores"], again)
print(f"rerun of seed 42 identical: {identical}")
assert identical, "not reproducible -- do not record"

rerun of seed 42 identical: True


### 3.5 Two ablations, and a question about class imbalance

3.67% of rows are claims. The standard reflex is to reweight the loss — `pos_weight`, or
oversample the minority class. **That reflex is imported from accuracy-oriented workflows and does
not transfer to a ranking metric.**

The argument is short. Under weighted BCE with positive weight *w*, the optimal prediction at
input *x* is `w·p(x) / (w·p(x) + 1 − p(x))`, which is a strictly increasing function of `p(x)` for
any *w* > 0. Gini depends only on rank order, and a strictly increasing transform preserves rank
order. So in the well-specified limit, reweighting **cannot change AUC at all** — it just relabels
the score axis. Whatever it does in practice is a finite-sample optimization artifact with no
guaranteed sign.

What *does* matter under this much imbalance is the **output bias initialization**. Starting the
output layer at `logit(base_rate) ≈ −3.27` rather than 0 saves the optimizer from spending its
first few hundred steps dragging a single scalar downward while the weight gradients are tiny.

Both are measured below rather than argued, plus a third check on a model nobody can accuse of
being mistuned: logistic regression with and without `class_weight="balanced"`.

In [10]:
ablations = {}
for name, over in [("pos_weight = 26.26", dict(pos_weight=float((1 - BASE_RATE) / BASE_RATE))),
                   ("no output-bias init", dict(init_output_bias=False))]:
    m, s, _ = cross_validate_model(nn_pipe(seed=42, **over), X, y, verbose=False)
    ablations[name] = (m, s)
    print(f"{name:<22} {m:+.5f} +/- {s:.5f}   "
          f"delta vs seed-42 baseline {m - per_seed[42]['mean']:+.5f}")

print("\nSame question on logistic regression -- does reweighting move a RANK metric?")
lr_kw = dict(max_iter=1000, random_state=SEED)
for label, kw in [("plain", lr_kw), ("class_weight='balanced'", dict(lr_kw, class_weight="balanced"))]:
    m, s, _ = cross_validate_model(
        make_pipeline(_build_preprocessor(len(cat_37), len(qty_37), scale=True),
                      LogisticRegression(**kw)), X, y, verbose=False)
    ablations[f"LR {label}"] = (m, s)
    print(f"  LR {label:<24} {m:+.5f} +/- {s:.5f}")

pos_weight = 26.26     +0.27269 +/- 0.00561   delta vs seed-42 baseline -0.00631


no output-bias init    +0.27043 +/- 0.00532   delta vs seed-42 baseline -0.00857

Same question on logistic regression -- does reweighting move a RANK metric?


  LR plain                    +0.25886 +/- 0.00441


  LR class_weight='balanced'  +0.25821 +/- 0.00405


### 3.6 The forest, re-run here

`results.md` records the forest at 0.27244 ± 0.00305, but that timing was measured on a different
machine on a different day. For the cost comparison to mean anything, both models have to be timed
in the same session on the same hardware — so the forest is re-run here rather than quoted.

In [11]:
def rf_pipe(n_jobs=-1):
    return make_pipeline(
        _build_preprocessor(len(cat_37), len(qty_37), scale=False),
        RandomForestClassifier(n_estimators=200, min_samples_leaf=50,
                               n_jobs=n_jobs, random_state=SEED))

t0 = time.perf_counter()
RF_MEAN, RF_STD, rf_scores = cross_validate_model(rf_pipe(), X, y, verbose=False)
RF_RUNTIME = time.perf_counter() - t0
print(f"forest  {RF_MEAN:+.5f} +/- {RF_STD:.5f}   ({RF_RUNTIME:.0f}s, n_jobs=-1)")
print(f"folds   {[f'{v:+.5f}' for v in rf_scores]}")
print(f"\nresults.md has 0.27244 +/- 0.00305 (#13) -- delta {RF_MEAN - 0.27244:+.5f}, "
      f"i.e. this reproduces the recorded run")

forest  +0.27248 +/- 0.00360   (115s, n_jobs=-1)
folds   ['+0.26674', '+0.27559', '+0.27303', '+0.27038', '+0.27667']

results.md has 0.27244 +/- 0.00305 (#13) -- delta +0.00004, i.e. this reproduces the recorded run


### 3.7 Capture rate at the riskiest 10%

Gini is the KPI, but it is abstract. #18 reported the number underwriting would actually feel:
of all the policyholders who went on to claim, what share landed in the riskiest 10% of the
ranking? The forest captured **21.09%** out of fold. Same definition here, same folds.

In [12]:
oof = cross_val_predict(nn_pipe(seed=42), X, y, cv=splitter, method="predict_proba")[:, 1]
pooled = gini_normalized(y, oof)
n_top = int(len(y) * 0.10)
capture = y[np.argsort(oof)[::-1][:n_top]].sum() / y.sum()

print(f"pooled out-of-fold Gini   {pooled:+.5f}   (CV mean for seed 42: {per_seed[42]['mean']:+.5f})")
print(f"riskiest 10%              {n_top:,} rows")
print(f"claims captured           {capture:.2%}   lift {capture/0.10:.2f}x over random")
print(f"forest, same definition   21.09%          delta {(capture - 0.2109)*100:+.2f}pp")

pooled out-of-fold Gini   +0.27734   (CV mean for seed 42: +0.27900)
riskiest 10%              47,596 rows
claims captured           21.49%   lift 2.15x over random
forest, same definition   21.09%          delta +0.40pp


### 3.8 Training and inference time

Rules, so the numbers mean something:

- **The `Runtime` column in `results.md` keeps its existing definition** — wall clock for the whole
  CV run including per-fold preprocessing. Everything below is a *breakdown*, not a redefinition.
- **Warm up and take a median of 3.** Consecutive runs of byte-identical work vary by ~20% on a
  laptop; a single sample is not a measurement.
- **Inference is reported as batch throughput at a stated batch size**, with single-row latency
  given separately. Scoring one row at a time measures Python dispatch overhead, not the model, and
  would distort the network by roughly 100×.
- **Parallelism is stated, not hidden.** The forest runs `n_jobs=-1` across 10 cores; the network
  runs on **one thread**. That is not a handicap — measured on this machine, 1 thread is *faster*
  than 6 for this workload (thread-pool synchronization dominates at this batch size) and produces
  bitwise-identical output. To make the comparison fair anyway, the forest is also fitted once with
  `n_jobs=1`.

In [13]:
tr_idx, va_idx = outer[0]
pre_nn = _build_preprocessor(len(cat_37), len(qty_37), scale=True)
t0 = time.perf_counter()
Xtr = pre_nn.fit_transform(X[tr_idx]); PRE_T = time.perf_counter() - t0
Xva = pre_nn.transform(X[va_idx])

def median_of(fn, k=3):
    fn()                                          # warm-up, discarded
    out = []
    for _ in range(k):
        t0 = time.perf_counter(); fn(); out.append(time.perf_counter() - t0)
    return float(np.median(out))

nn_bare = TorchMLPClassifier(**BEST_CFG, epochs=EPOCHS, random_state=SEED)
NN_FIT = median_of(lambda: nn_bare.fit(Xtr, y[tr_idx]))
NN_INF = median_of(lambda: nn_bare.predict_proba(Xva))

pre_rf = _build_preprocessor(len(cat_37), len(qty_37), scale=False)
Xtr_rf, Xva_rf = pre_rf.fit_transform(X[tr_idx]), pre_rf.transform(X[va_idx])
rf_bare = RandomForestClassifier(n_estimators=200, min_samples_leaf=50,
                                 n_jobs=-1, random_state=SEED)
t0 = time.perf_counter(); rf_bare.fit(Xtr_rf, y[tr_idx]); RF_FIT = time.perf_counter() - t0
RF_INF = median_of(lambda: rf_bare.predict_proba(Xva_rf))

print(f"preprocessing   {PRE_T:6.2f}s   (fit_transform, {len(tr_idx):,} rows -- same for both)")
print(f"\n{'':16}{'fit':>10}{'inference':>12}{'us/row':>10}{'rows/s':>14}")
print(f"{'neural net':<16}{NN_FIT:>9.2f}s{NN_INF:>11.3f}s"
      f"{NN_INF/len(va_idx)*1e6:>10.2f}{len(va_idx)/NN_INF:>14,.0f}")
print(f"{'random forest':<16}{RF_FIT:>9.2f}s{RF_INF:>11.3f}s"
      f"{RF_INF/len(va_idx)*1e6:>10.2f}{len(va_idx)/RF_INF:>14,.0f}")
print(f"\ninference: the network is {RF_INF/NN_INF:.0f}x faster on {len(va_idx):,} rows "
      f"(batch size 8192)")
print(f"fit:       {NN_FIT/EPOCHS:.2f}s per epoch x {EPOCHS} epochs, single thread")

preprocessing     1.68s   (fit_transform, 380,773 rows -- same for both)

                       fit   inference    us/row        rows/s
neural net           5.21s      0.014s      0.15     6,595,427
random forest       21.13s      0.395s      4.15       240,976

inference: the network is 27x faster on 95,194 rows (batch size 8192)
fit:       0.58s per epoch x 9 epochs, single thread


In [14]:
# Single-row latency, kept separate from throughput on purpose.
NN_ONE = median_of(lambda: nn_bare.predict_proba(Xva[:1]), 5)
RF_ONE = median_of(lambda: rf_bare.predict_proba(Xva_rf[:1]), 5)
print(f"single-row latency   network {NN_ONE*1e3:6.3f}ms    forest {RF_ONE*1e3:6.3f}ms")

# Controlled single-core fit, so the core-seconds claim is measured not asserted.
t0 = time.perf_counter()
RandomForestClassifier(n_estimators=200, min_samples_leaf=50, n_jobs=1,
                       random_state=SEED).fit(Xtr_rf, y[tr_idx])
RF_FIT_1C = time.perf_counter() - t0
print(f"\nsingle-core fit, one fold:")
print(f"  forest  (n_jobs=1)   {RF_FIT_1C:7.1f}s")
print(f"  network (1 thread)   {NN_FIT:7.1f}s")
print(f"  the forest costs {RF_FIT_1C/NN_FIT:.0f}x the network's compute for the same fold")

single-row latency   network  0.045ms    forest 16.510ms



single-core fit, one fold:
  forest  (n_jobs=1)      98.5s
  network (1 thread)       5.2s
  the forest costs 19x the network's compute for the same fold


---
## 4. Head to head

Everything above, in one place. The Gini column applies the project's standing rule: a delta only
counts if it exceeds its own row's fold standard deviation.

In [15]:
delta = NN_MEAN - RF_MEAN
noise = max(NN_FOLD_SD, RF_STD, NN_SEED_SD)

rows = [
    ("Primary metric (Gini)", f"{NN_MEAN:+.5f} +/- {NN_FOLD_SD:.5f}",
                              f"{RF_MEAN:+.5f} +/- {RF_STD:.5f}"),
    ("Capture @ riskiest 10%", f"{capture:.2%}", "21.09%"),
    ("Training time (5-fold CV)", f"{NN_RUNTIME/3:.0f}s", f"{RF_RUNTIME:.0f}s"),
    ("Training, one fold, 1 core", f"{NN_FIT:.0f}s", f"{RF_FIT_1C:.0f}s"),
    ("Inference (batch, per row)", f"{NN_INF/len(va_idx)*1e6:.2f} us",
                                   f"{RF_INF/len(va_idx)*1e6:.2f} us"),
    ("Inference (rows/sec)", f"{len(va_idx)/NN_INF:,.0f}", f"{len(va_idx)/RF_INF:,.0f}"),
    ("Data volume used", f"{len(X):,} rows / {int(y.sum()):,} claims",
                         f"{len(X):,} rows / {int(y.sum()):,} claims"),
    ("Features seen", f"{N_FEATURES} (one-hot, scaled)", f"{N_FEATURES} (one-hot, unscaled)"),
]
w = max(len(r[0]) for r in rows)
print(f"{'':{w}}  {'NEURAL NET':>26}  {'RANDOM FOREST':>26}")
print("-" * (w + 56))
for label, a, b in rows:
    print(f"{label:<{w}}  {a:>26}  {b:>26}")

print("\n" + "-" * (w + 56))
print(f"Gini delta (NN - forest): {delta:+.5f}")
print(f"Largest relevant noise:   {noise:.5f}  "
      f"(max of NN fold sd, forest fold sd, NN seed spread)")
print(f"Verdict on the metric:    {'SEPARABLE' if abs(delta) > noise else 'INDISTINGUISHABLE'}"
      f" -- |delta| is {abs(delta)/noise:.2f}x the noise")

                                            NEURAL NET               RANDOM FOREST
----------------------------------------------------------------------------------
Primary metric (Gini)             +0.27801 +/- 0.00678        +0.27248 +/- 0.00360
Capture @ riskiest 10%                          21.49%                      21.09%
Training time (5-fold CV)                          35s                        115s
Training, one fold, 1 core                          5s                         99s
Inference (batch, per row)                     0.15 us                     4.15 us
Inference (rows/sec)                         6,595,427                     240,976
Data volume used            475,967 rows / 17,461 claims  475,967 rows / 17,461 claims
Features seen                    206 (one-hot, scaled)     206 (one-hot, unscaled)

----------------------------------------------------------------------------------
Gini delta (NN - forest): +0.00552
Largest relevant noise:   0.00678  (max of NN f

### 4.1 Paired comparison — the rule and the better test disagree

The project's standing rule is *"a delta counts only if it exceeds this row's fold std"*. That rule
is what rejected #17 and #22, so it applies here without special pleading. By it, the network's
delta does **not** clear its own fold std, and the result is inconclusive.

But that rule is an **unpaired** heuristic, and it throws away the single most useful fact about
this comparison: both models ran on the *identical five folds*. Fold 1 is hard for both of them;
fold 2 is easy for both. Comparing a difference against the spread *across* folds counts that
shared fold-difficulty as noise, when it cancels exactly.

The paired version — compare the two models fold by fold, then look at the spread of the
*differences* — is the same test #13 reached for when it reported "4 of 5 folds improved, none got
worse." It is strictly more sensitive, and it is the honest test when the folds are shared.

Both are reported below. Where they disagree, the paired result is the more informative one and the
unpaired rule is the more conservative one.

In [16]:
nn_per_fold = np.mean([per_seed[s]["scores"] for s in (42, 43, 44)], axis=0)
paired = nn_per_fold - rf_scores

print(f"{'fold':<6}{'network':>10}{'forest':>10}{'delta':>10}")
for i, (a, b, d) in enumerate(zip(nn_per_fold, rf_scores, paired), 1):
    print(f"{i:<6}{a:>+10.5f}{b:>+10.5f}{d:>+10.5f}")

se = paired.std(ddof=1) / np.sqrt(len(paired))
print(f"\nfolds improved      {(paired > 0).sum()} of {len(paired)}")
print(f"paired mean delta   {paired.mean():+.5f}")
print(f"spread of deltas    {paired.std(ddof=1):.5f}   (std error of the mean {se:.5f})")
print(f"t statistic (4 df)  {paired.mean()/se:.2f}")

print(f"\nUNPAIRED (the project's standing rule):")
print(f"  delta {NN_MEAN - RF_MEAN:+.5f}  vs this row's fold std {NN_FOLD_SD:.5f}"
      f"  ->  {'clears' if NN_MEAN - RF_MEAN > NN_FOLD_SD else 'does NOT clear'}")
print(f"PAIRED (shared folds, the more sensitive test):")
print(f"  delta {paired.mean():+.5f}  vs std error {se:.5f}"
      f"  ->  {'separable' if abs(paired.mean()) > 2*se else 'not separable'}")
print(f"\nrelative improvement: {(NN_MEAN - RF_MEAN)/RF_MEAN:+.2%} of the forest's Gini")

fold     network    forest     delta
1       +0.26831  +0.26674  +0.00157
2       +0.28726  +0.27559  +0.01167
3       +0.27766  +0.27303  +0.00463
4       +0.27561  +0.27038  +0.00523
5       +0.28121  +0.27667  +0.00454

folds improved      5 of 5
paired mean delta   +0.00552
spread of deltas    0.00372   (std error of the mean 0.00166)
t statistic (4 df)  3.32

UNPAIRED (the project's standing rule):
  delta +0.00552  vs this row's fold std 0.00678  ->  does NOT clear
PAIRED (shared folds, the more sensitive test):
  delta +0.00552  vs std error 0.00166  ->  separable

relative improvement: +2.03% of the forest's Gini


### 4.2 The two dimensions that aren't measurable

**Interpretability — forest 3/5, network 1/5.**

The forest is not a glass box, but it answers useful questions. `feature_importances_` ranks
columns; permutation importance gives a defensible per-column contribution and was the entire basis
for #13 and #17; individual trees can be printed and read. That is what made the calc-column
ablation possible in the first place. It loses points because 200 trees cannot be reasoned about as
a whole and it gives no per-applicant explanation without extra tooling.

The network scores 1 because nothing above is available. There is no native importance, the
learned representation is 128 anonymous units over 206 already-anonymized inputs, and any
explanation requires bolting on SHAP or integrated gradients — more code, more compute, and an
approximation rather than the model's own answer. For a model that would feed **insurance pricing**,
where the README already flags proxy discrimination as a deployment blocker, dropping from 3 to 1
is not a cosmetic loss. It removes the audit surface on the axis regulators care about.

**Engineering effort — forest ~2h, network ~8h.**

The forest was three lines inside an existing pipeline, plus the ablation that found the 37-column
set. Effectively all of its cost was analysis, not plumbing.

The network's ~8 hours went almost entirely to things that had nothing to do with modelling: adding
a 121MB dependency to a shared lockfile; writing an estimator that survives `clone()` (where most
mistakes fail *silently* — see section 2); establishing that a diverged run returns a large *finite*
loss and so needs an explicit threshold, not just a `isfinite` check; proving bitwise
reproducibility; designing the tuning protocol so the epoch budget isn't fitted to noise; and
converting that budget between two training-set sizes. The actual network is about fifteen lines.

That ratio is the honest finding. **It is also a one-time cost** — the second network on this
codebase would be much cheaper, which matters if the answer here is "not yet, but maybe later."